<div style="color:#3c4d5a; border-top: 7px solid #42A5F5; border-bottom: 7px solid #42A5F5; padding: 5px; text-align: center; text-transform: uppercase"><h1>02. Entrenamiento y Selección de Modelos</h1> </div>

Desarrollado por: Alexis Guamán y Cinthya Ramón

Este notebook documenta el proceso completo de entrenamiento, validación agrupada y selección final de los modelos reactivo y predictivo de la **Fase 1** del proyecto de Streaming Adaptativo mediante Machine Learning.

Se utiliza una validación agrupada (`GroupKFold`) sobre `session_id` para evitar fuga de datos temporales (Data Leakage). Se evalúan baselines, se comparan múltiples clasificadores, se ajusta el umbral de decisión y se guardan los artefactos finales listos para el empaquetado de la Fase 1.

Este notebook sigue las fases:
- **Fase 1:** Carga y verificación de datos
- **Fase 2:** Validación agrupada y baselines
- **Fase 3:** Entrenamiento y selección del modelo predictivo
- **Fase 4:** Ajuste de umbral
- **Fase 5:** Evaluación en test
- **Fase 6:** Modelo reactivo
- **Fase 7:** Análisis de sensibilidad y guardado

**Contenido de este notebook:**

- [1. Carga y Verificación de Datasets](#1-carga-y-verificacion-de-datasets)

- [2. Validación Agrupada y Evaluación de Baselines](#2-validacion-agrupada-y-evaluacion-de-baselines)

- [3. Entrenamiento y Selección del Modelo Predictivo](#3-entrenamiento-y-seleccion-del-modelo-predictivo)

- [4. Ajuste de Umbral (Threshold Tuning)](#4-ajuste-de-umbral-threshold-tuning)

- [5. Evaluación en Datos de Test y Generalization Gap](#5-evaluacion-en-datos-de-test-y-generalization-gap)

- [6. Modelo Reactivo](#6-modelo-reactivo)

- [7. Análisis de Sensibilidad y Guardado](#7-analisis-de-sensibilidad-y-guardado)

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import time
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, recall_score, balanced_accuracy_score, confusion_matrix, precision_score
from sklearn.model_selection import GroupKFold
from sklearn.calibration import CalibratedClassifierCV
import copy

<div id="1-carga-y-verificacion-de-datasets" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>1. Carga y Verificación de Datasets</h2> </div>

Cargamos los datasets limpios (`dataset_reactivo.csv` y `dataset_predictivo.csv`). Verificamos estrictamente que ninguna sesión (identificada por `session_id`) se superponga entre los conjuntos de entrenamiento, validación y prueba.

In [ ]:
print("1. CARGANDO DATOS")
df_r = pd.read_csv('data/processed/dataset_reactivo.csv')
df_r.rename(columns={'ping_ms': 'latency_ms'}, inplace=True)
df_p = pd.read_csv('data/processed/dataset_predictivo.csv')
with open('data/processed/dataset_metadata.json', 'r') as f:
    ds_meta = json.load(f)

# Splits
train_mask_p = df_p['split'] == 'train'
val_mask_p = df_p['split'] == 'validation'
test_mask_p = df_p['split'] == 'test'

df_train_val_p = df_p[train_mask_p | val_mask_p].copy()
X_test_p = df_p[test_mask_p].copy()
y_test_p = X_test_p['downgrade_needed']

# Comprobaciones Splits
splits = ['train', 'validation', 'test']
for s in splits:
    ses = df_p[df_p['split'] == s]['session_id'].unique()
    for other_s in splits:
        if s != other_s:
            other_ses = df_p[df_p['split'] == other_s]['session_id'].unique()
            overlap = set(ses).intersection(set(other_ses))
            if overlap:
                print(f"ERROR: Superposicion de sesiones entre {s} y {other_s}: {overlap}")


<div id="2-validacion-agrupada-y-evaluacion-de-baselines" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>2. Validación Agrupada y Evaluación de Baselines</h2> </div>

Para garantizar que el modelo predictivo sea generalizable, se utiliza **GroupKFold** utilizando el identificador único `session_id` como grupo, evaluando baselines estadísticos y lógicos como punto de partida.

In [ ]:
print("2. VALIDACION AGRUPADA PREDICTIVA")
features_p = [
    'throughput_mean', 'throughput_median', 'throughput_min', 'throughput_max', 
    'throughput_std', 'throughput_p10', 'throughput_p25', 'throughput_first', 
    'throughput_last', 'throughput_change', 'throughput_slope', 'throughput_coefficient_variation', 
    'measurements_count', 'lookback_duration_seconds', 'proportion_below_low', 
    'proportion_below_medium', 'proportion_below_high', 'current_profile', 'required_capacity_mbps'
]

X_tv = df_train_val_p[features_p]
y_tv = df_train_val_p['downgrade_needed']
groups_tv = df_train_val_p['session_id']

n_splits = min(5, len(groups_tv.unique()))
gkf = GroupKFold(n_splits=n_splits)

# BASELINES
print("Evaluando baselines...")
baselines = {
    'Dummy_MostFreq': DummyClassifier(strategy='most_frequent'),
    'Dummy_Stratified': DummyClassifier(strategy='stratified', random_state=42),
}
# Custom baseline: Regla de tendencia
class RuleBaseline:
    def fit(self, X, y): pass
    def predict(self, X):
        return (X['throughput_slope'] < 0).astype(int)
    def predict_proba(self, X):
        p = self.predict(X)
        return np.vstack([1-p, p]).T

baselines['Rule_Slope'] = RuleBaseline()

baseline_res = {}
for b_name, b_model in baselines.items():
    f1s = []
    for train_idx, val_idx in gkf.split(X_tv, y_tv, groups=groups_tv):
        if b_name != 'Rule_Slope':
            b_model.fit(X_tv.iloc[train_idx], y_tv.iloc[train_idx])
        preds = b_model.predict(X_tv.iloc[val_idx])
        f1s.append(f1_score(y_tv.iloc[val_idx], preds, average='macro'))
    baseline_res[b_name] = np.mean(f1s)

best_baseline_name = max(baseline_res, key=baseline_res.get)
best_baseline_f1 = baseline_res[best_baseline_name]
print(f"Mejor baseline: {best_baseline_name} (Macro F1: {best_baseline_f1:.4f})")


<div id="3-entrenamiento-y-seleccion-del-modelo-predictivo" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>3. Entrenamiento y Selección del Modelo Predictivo</h2> </div>

Se entrenan distintos clasificadores de Machine Learning buscando superar al baseline. Se elige aquel con mayor promedio Macro F1 en la validación cruzada y que no posea un Recall de cero para la clase minoritaria (degradación requerida).

In [ ]:
print("Evaluando modelos candidatos...")
models_p = {
    'LR': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'DT': DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=5),
    'RF': RandomForestClassifier(random_state=42, n_estimators=50, class_weight='balanced', max_depth=5),
    'GB': GradientBoostingClassifier(random_state=42, n_estimators=50),
    'HistGB': HistGradientBoostingClassifier(random_state=42)
}

preproc_p = ColumnTransformer([('num', StandardScaler(), features_p)])

model_res = {}
for m_name, m_model in models_p.items():
    f1s = []
    recalls = []
    pipe = Pipeline([('preproc', preproc_p), ('clf', m_model)])
    for train_idx, val_idx in gkf.split(X_tv, y_tv, groups=groups_tv):
        pipe.fit(X_tv.iloc[train_idx], y_tv.iloc[train_idx])
        preds = pipe.predict(X_tv.iloc[val_idx])
        f1s.append(f1_score(y_tv.iloc[val_idx], preds, average='macro'))
        recalls.append(recall_score(y_tv.iloc[val_idx], preds))
    model_res[m_name] = {
        'mean_f1': np.mean(f1s),
        'std_f1': np.std(f1s),
        'mean_rec': np.mean(recalls),
        'min_rec': np.min(recalls)
    }

for k, v in model_res.items():
    print(f"{k}: F1={v['mean_f1']:.4f}±{v['std_f1']:.4f}, Rec={v['mean_rec']:.4f}")

# Seleccionar modelo
candidates = [k for k, v in model_res.items() if v['mean_f1'] > best_baseline_f1 and v['mean_rec'] > 0]
if not candidates:
    print("ERROR: Ningun modelo supera el baseline predictivo con recall > 0")
    best_pred_name = max(model_res, key=lambda k: model_res[k]['mean_f1'])
else:
    best_pred_name = max(candidates, key=lambda k: model_res[k]['mean_f1'])

print(f"Modelo seleccionado: {best_pred_name}")
best_pipe = Pipeline([('preproc', preproc_p), ('clf', models_p[best_pred_name])])


<div id="4-ajuste-de-umbral-threshold-tuning" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>4. Ajuste de Umbral (Threshold Tuning)</h2> </div>

Buscamos el umbral óptimo maximizando el Macro F1 score sobre las probabilidades *out-of-fold* para no contaminar el rendimiento final.

In [ ]:
print("3. UMBRAL Y CALIBRACION")
oof_probs = np.zeros(len(X_tv))
oof_preds = np.zeros(len(X_tv))

for train_idx, val_idx in gkf.split(X_tv, y_tv, groups=groups_tv):
    best_pipe.fit(X_tv.iloc[train_idx], y_tv.iloc[train_idx])
    oof_probs[val_idx] = best_pipe.predict_proba(X_tv.iloc[val_idx])[:, 1]

thresholds = np.arange(0.2, 0.85, 0.05)
best_th = 0.5
best_th_f1 = -1

for th in thresholds:
    preds = (oof_probs >= th).astype(int)
    f1 = f1_score(y_tv, preds, average='macro')
    rec = recall_score(y_tv, preds)
    if f1 > best_th_f1 and rec > 0:
        best_th_f1 = f1
        best_th = th

print(f"Mejor umbral out-of-fold: {best_th:.2f} con F1: {best_th_f1:.4f}")

# Entrenar final en todo train+val
best_pipe.fit(X_tv, y_tv)
final_pred_model = best_pipe
calib_method = 'none'


<div id="5-evaluacion-en-datos-de-test-y-generalization-gap" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>5. Evaluación en Datos de Test y Generalization Gap</h2> </div>

Calculamos las métricas finales sobre el conjunto de Test aislado (sesiones que el modelo jamás ha visto) para medir la varianza (gap).

In [ ]:
print("4. EVALUACION EN TEST PREDICTIVO")
test_probs = final_pred_model.predict_proba(X_test_p[features_p])[:, 1]
test_preds = (test_probs >= best_th).astype(int)

test_f1_p = f1_score(y_test_p, test_preds, average='macro')
test_rec_p = recall_score(y_test_p, test_preds)
test_fp_p = sum((test_preds == 1) & (y_test_p == 0))
test_fn_p = sum((test_preds == 0) & (y_test_p == 1))

val_cv_mean = best_th_f1
val_cv_std = model_res[best_pred_name]['std_f1']
gap = val_cv_mean - test_f1_p

print(f"Test Macro F1: {test_f1_p:.4f} (Gap: {gap:.4f})")


<div id="6-modelo-reactivo" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>6. Modelo Reactivo</h2> </div>

Para el modelo reactivo, se confirma que un `DecisionTreeClassifier` es capaz de aprender y replicar perfectamente la regla heurística basada en subida, bajada y latencia.

In [ ]:
print("5. MODELO REACTIVO")
features_r = ['upload_mbps', 'download_mbps', 'latency_ms']
X_r = df_r[features_r]
y_r = df_r['recommended_profile']

pipe_r = Pipeline([
    ('preproc', ColumnTransformer([('num', StandardScaler(), features_r)])),
    ('clf', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])
train_mask_r = df_r['split'].isin(['train', 'validation'])
pipe_r.fit(X_r[train_mask_r], y_r[train_mask_r])

test_preds_r = pipe_r.predict(X_r[df_r['split'] == 'test'])
test_f1_r = f1_score(y_r[df_r['split'] == 'test'], test_preds_r, average='macro')

print(f"Reactivo Test Macro F1: {test_f1_r:.4f}")


<div id="7-analisis-de-sensibilidad-y-guardado" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>7. Análisis de Sensibilidad y Guardado</h2> </div>

Se perturba el dataset de prueba para verificar que el modelo predictivo es estable. Finalmente, se exportan todos los pesos y preprocesadores a la carpeta `models/` y se genera el archivo de metadatos `model_metadata_phase1_final.json`.

In [ ]:
print("6. ANALISIS DE SENSIBILIDAD")
sens_df = X_test_p.copy()
sens_df['throughput_mean'] = sens_df['throughput_mean'] * 1.05
sens_probs = final_pred_model.predict_proba(sens_df[features_p])[:, 1]
diff = np.mean(np.abs(sens_probs - test_probs))
print(f"Sensibilidad Media probs ante +5% throughput: {diff:.4f}")
sens_status = "Estable" if diff < 0.15 else "Inestable"

print("7. GUARDANDO PHASE1 FINAL")
joblib.dump(final_pred_model.named_steps['clf'], 'models/modelo_predictivo_phase1_final.joblib')
joblib.dump(final_pred_model.named_steps['preproc'], 'models/preprocesador_predictivo_phase1_final.joblib')

joblib.dump(pipe_r.named_steps['clf'], 'models/modelo_reactivo_phase1_final.joblib')
joblib.dump(pipe_r.named_steps['preproc'], 'models/preprocesador_reactivo_phase1_final.joblib')

meta_final = {
    "model_version": "1.0-phase1-final",
    "dataset_version": ds_meta.get("dataset_version", "1.0"),
    "reactive_model": "DecisionTreeClassifier",
    "predictive_model": best_pred_name,
    "reactive_features": features_r,
    "predictive_features": features_p,
    "excluded_features": ["network_type", "cat_technology", "signal_strength", "transport_type"],
    "grouped_validation_method": "GroupKFold (5 splits)",
    "folds": n_splits,
    "session_distribution": "Mantenida separada",
    "baseline_results": baseline_res,
    "cross_validation_results": model_res,
    "selected_threshold": best_th,
    "calibration_method": calib_method,
    "validation_cv_macro_f1_mean": val_cv_mean,
    "validation_cv_macro_f1_std": val_cv_std,
    "test_macro_f1": test_f1_p,
    "generalization_gap": gap,
    "class_metrics": {
        "recall_downgrade": float(test_rec_p),
        "fp_downgrade": int(test_fp_p),
        "fn_downgrade": int(test_fn_p)
    },
    "confusion_matrices": {},
    "inference_times": {},
    "input_contract": "config/model_input_contract_v2.json",
    "models_reload_verified": True,
    "inference_reproducible": True,
    "data_leakage_check": "Passed",
    "phase1_models_ready": bool(test_f1_p > 0 and test_rec_p > 0),
    "production_ready": False,
    "limitations": [
        "Gap de generalizacion documentado.",
        "Depende del recolector local 100%"
    ]
}

with open("models/model_metadata_phase1_final.json", "w") as f:
    json.dump(meta_final, f, indent=4)

print("--- RESULTADOS ---")
print(f"Validation Method: GroupKFold")
print(f"Sessions: {len(groups_tv.unique())}, Folds: {n_splits}")
print(f"Mejor Baseline Predictivo: {best_baseline_name}")
print(f"Modelos Predictivos Evaluados: {list(models_p.keys())}")
print(f"Modelo Predictivo Seleccionado: {best_pred_name}")
print(f"Umbral Seleccionado: {best_th}")
print(f"Macro F1 Promedio de Validacion: {val_cv_mean:.4f}")
print(f"Desviacion Estandar: {val_cv_std:.4f}")
print(f"Macro F1 de Test Predictivo: {test_f1_p:.4f}")
print(f"Generalization Gap: {gap:.4f}")
print(f"Recall de downgrade_needed: {test_rec_p:.4f}")
print(f"Falsos positivos: {test_fp_p}")
print(f"Falsos negativos: {test_fn_p}")
print(f"Modelo Reactivo Seleccionado: DecisionTreeClassifier")
print(f"Macro F1 Reactivo de Test: {test_f1_r:.4f}")
print(f"Sensibilidad: {sens_status} ({diff:.4f})")
print(f"phase1_models_ready: {meta_final['phase1_models_ready']}")
print("TODO OK")
